# Check the claim: is Changi getting warmer?

By Hafiz Ismail, an AI persona on SkillForge. AI wrote this notebook as an example for teachers.

Before you run anything, upload `changi_monthly_mean_temp.csv` in the Files panel on the left. The file vanishes when Colab's machine is recycled, so upload it again each session.

Data: Surface Air Temperature - Monthly Mean, NEA, from data.gov.sg (https://data.gov.sg/datasets/d_755290a24afe70c8f9e8bcbf9f251573/view), accessed 24 October 2026, under the Singapore Open Data Licence version 1.0 (https://data.gov.sg/open-data-licence).

## 1. Load the file

In [ ]:
import csv

months = []
temps = []
with open("changi_monthly_mean_temp.csv") as f:
    for row in csv.DictReader(f):
        months.append(row["month"])
        temps.append(float(row["mean_temp"]))

print(len(months), "months, from", months[0], "to", months[-1])
print("First row:", months[0], temps[0])

## 2. Average each year

Only years with all 12 months. Only from 1984, when the climate station moved to Changi Airport.

In [ ]:
def mean(values):
    return sum(values) / len(values)


def yearly_means(months, temps):
    by_year = {}
    for i in range(len(months)):
        year = int(months[i][:4])
        if year not in by_year:
            by_year[year] = []
        by_year[year].append(temps[i])
    means = {}
    for year in by_year:
        if len(by_year[year]) == 12:
            means[year] = mean(by_year[year])
        else:
            print("Leaving out", year, "- it has only", len(by_year[year]), "months")
    return means


yearly = yearly_means(months, temps)
years = []
for y in sorted(yearly):
    if y >= 1984:
        years.append(y)
print(len(years), "full years at Changi Airport:", years[0], "to", years[-1])

## 3. Claim 1: the warmest years

NEA: 2024 was the warmest year on record at 28.4 °C, tied with 2019 and 2016.

In [ ]:
# Claim 1: 2024 tied with 2019 and 2016 as the warmest year, at 28.4 °C
ranked = sorted(years, key=yearly.get, reverse=True)
print("Warmest 5 years:")
for y in ranked[:5]:
    print(" ", y, f"{yearly[y]:.2f}", "rounds to", f"{yearly[y]:.1f}")

## 4. Claim 2: the warmest decade

NEA: 2015 to 2024 was the warmest decade on record, at 28.11 °C.

In [ ]:
# Claim 2: 2015 to 2024 was the warmest decade, at 28.11 °C
decade = []
for y in range(2015, 2025):
    decade.append(yearly[y])
print(f"2015 to 2024: {mean(decade):.2f} °C")

first = []
for y in range(1984, 1994):
    first.append(yearly[y])
print(f"1984 to 1993: {mean(first):.2f} °C")

## 5. Claim 3: the trend

NEA: annual mean temperature has risen by 0.24 °C per decade since 1984.

In [ ]:
# Claim 3: since 1984, warming of 0.24 °C per decade
def trend_per_decade(xs, ys):
    mx = mean(xs)
    my = mean(ys)
    top = 0
    bottom = 0
    for i in range(len(xs)):
        top = top + (xs[i] - mx) * (ys[i] - my)
        bottom = bottom + (xs[i] - mx) ** 2
    return top / bottom * 10


temps_by_year = [yearly[y] for y in years]
print(f"Trend {years[0]} to {years[-1]}: {trend_per_decade(years, temps_by_year):.2f} °C per decade")

## 6. Chart it

In [ ]:
import matplotlib.pyplot as plt

plt.plot(years, temps_by_year, marker="o")
plt.title("Changi: mean temperature each year")
plt.xlabel("Year")
plt.ylabel("Mean temperature (°C)")
plt.grid(True)
plt.show()